<a href="https://colab.research.google.com/github/vandit-11/learning-data-science/blob/main/Langgraph_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# -----------------------------------------
# 1. DEFINE THE STATE
# This represents the data passed between nodes
# -----------------------------------------
class MathState(TypedDict):
    a: int        # First number
    b: int        # Second number
    result: float # The running total

# -----------------------------------------
# 2. DEFINE THE NODES (The Actions)
# Each node takes the state, performs an action,
# and returns the updated part of the state.
# -----------------------------------------
def add_node(state: MathState):
    res = state["a"] + state["b"]
    print(f"Node [Add]: {state['a']} + {state['b']} = {res}")
    return {"result": res}

def multiply_node(state: MathState):
    res = state["result"] * 2
    print(f"Node [Multiply]: {state['result']} * 2 = {res}")
    return {"result": res}

def divide_node(state: MathState):
    res = state["result"] / 3
    print(f"Node [Divide]: {state['result']} / 3 = {res}")
    return {"result": res}

def square_node(state: MathState):
    res = state["result"] ** 2
    print(f"Node [Square]: {state['result']}^2 = {res}")
    return {"result": res}

# -----------------------------------------
# 3. DEFINE THE CONDITIONAL LOGIC
# This function decides which node to go to next
# -----------------------------------------
def check_divisibility(state: MathState) -> str:
    # If divisible by 3, return the string "divide"
    if state["result"] % 3 == 0:
        print(f"Condition: {state['result']} IS divisible by 3. Routing to Divide.")
        return "divide"
    # Otherwise, return the string "square"
    else:
        print(f"Condition: {state['result']} IS NOT divisible by 3. Routing to Square.")
        return "square"

# -----------------------------------------
# 4. BUILD THE GRAPH
# -----------------------------------------
# Initialize the graph with our state structure
workflow = StateGraph(MathState)

# Add all our nodes to the graph
workflow.add_node("add", add_node)
workflow.add_node("multiply", multiply_node)
workflow.add_node("divide", divide_node)
workflow.add_node("square", square_node)

# Define the flow (Edges)
workflow.add_edge(START, "add")       # Step 1: Start -> Add
workflow.add_edge("add", "multiply")  # Step 2: Add -> Multiply

# Step 3: Conditional Routing
workflow.add_conditional_edges(
    "multiply",               # The node we are coming from
    check_divisibility,       # The function that makes the decision
    {
        "divide": "divide",   # If function returns "divide", go to divide node
        "square": "square"    # If function returns "square", go to square node
    }
)

# Step 4: Finish the paths
workflow.add_edge("divide", "square") # If we divided, we must square next
workflow.add_edge("square", END)      # After squaring, the workflow ends

# Compile the graph into an executable application
app = workflow.compile()

In [ ]:
print("--- SCENARIO 1 ---")
initial_state_1 = {"a": 1, "b": 2, "result": 0}
final_state_1 = app.invoke(initial_state_1)
print(f"Final Output: {final_state_1['result']}\n")

--- SCENARIO 1 ---
Node [Add]: 1 + 2 = 3
Node [Multiply]: 3 * 2 = 6
Condition: 6 IS divisible by 3. Routing to Divide.
Node [Divide]: 6 / 3 = 2.0
Node [Square]: 2.0^2 = 4.0
Final Output: 4.0



In [ ]:
print("--- SCENARIO 2 ---")
initial_state_2 = {"a": 2, "b": 2, "result": 0}
final_state_2 = app.invoke(initial_state_2)
print(f"Final Output: {final_state_2['result']}\n")

--- SCENARIO 2 ---
Node [Add]: 2 + 2 = 4
Node [Multiply]: 4 * 2 = 8
Condition: 8 IS NOT divisible by 3. Routing to Square.
Node [Square]: 8^2 = 64
Final Output: 64



#Prod-ready AI Agent

In [ ]:
!pip install -qU langgraph langchain-openai langchain-core pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.9/471.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 64.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.0 which is incompatible.


In [ ]:
import os
from typing import TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Initialize our placeholder AI (Temperature 0 for deterministic, strict outputs)
# In production, you might use AzureChatOpenAI or a locally hosted model
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ==========================================
# 1. DEFINE PYDANTIC MODELS (Strict AI Outputs)
# ==========================================
class DraftOutput(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning for the decision.")
    proposed_refund_amount: float = Field(description="The exact dollar amount to refund. 0.0 if no refund.")
    draft_email: str = Field(description="The exact email text to send to the customer.")

class ComplianceOutput(BaseModel):
    reasoning: str = Field(description="Why the draft passed or failed.")
    is_compliant: bool = Field(description="True if safe to send, False if it violates policy.")
    feedback_for_writer: str = Field(description="If False, tell the writer how to fix it.")

# ==========================================
# 2. DEFINE THE GRAPH STATE
# ==========================================
class TicketState(TypedDict):
    ticket_id: str
    customer_issue: str
    crm_data: dict
    # AI generated fields
    draft_reasoning: str
    proposed_refund: float
    draft_email: str
    compliance_passed: bool
    compliance_feedback: str
    revision_count: int

# ==========================================
# 3. PRODUCTION PROMPTS & AI NODES
# ==========================================
def fetch_data_node(state: TicketState):
    print("🔄 SYSTEM: Fetching CRM & Order Data...")
    # Placeholder for database/API query
    return {
        "crm_data": {
            "order_number": "ORD-12345",
            "item_name": "Premium Espresso Machine",
            "item_cost": 250.00,
            "customer_lifetime_value": 1200.00,
            "warranty_active": True
        },
        "revision_count": 0 # Initialize loop counter
    }

def adjudicate_and_draft_node(state: TicketState):
    print(f"🧠 AI (Adjudicator): Analyzing ticket and drafting response... (Revision: {state.get('revision_count')})")

    system_prompt = """You are a Senior Customer Support Adjudicator for a premium e-commerce brand.
    Your job is to read the customer's issue, look at their CRM data, and resolve the ticket.

    POLICIES:
    1. If the item is broken and under warranty, offer a full refund up to the 'item_cost'.
    2. NEVER offer a refund greater than the 'item_cost'.
    3. Maintain a highly empathetic, professional, and apologetic tone.
    4. If the customer is highly valuable (LTV > $1000), prioritize speed and empathy.
    """

    user_prompt = """
    TICKET INFO:
    Customer Issue: {issue}
    CRM Data: {crm}

    {compliance_feedback}
    """

    # Inject previous feedback if the compliance node rejected an earlier draft
    feedback_str = ""
    if state.get("compliance_feedback"):
        feedback_str = f"PREVIOUS ATTEMPT FAILED COMPLIANCE. FIX THIS: {state['compliance_feedback']}"

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", user_prompt)
    ])

    # Bind the Pydantic model to force the LLM to output valid JSON
    structured_llm = llm.with_structured_output(DraftOutput)
    chain = prompt | structured_llm

    # Execute AI
    result: DraftOutput = chain.invoke({
        "issue": state["customer_issue"],
        "crm": state["crm_data"],
        "compliance_feedback": feedback_str
    })

    return {
        "draft_reasoning": result.reasoning,
        "proposed_refund": result.proposed_refund_amount,
        "draft_email": result.draft_email,
        "revision_count": state["revision_count"] + 1
    }

def compliance_node(state: TicketState):
    print("🛡️ AI (Compliance Auditor): Checking draft against company policy...")

    system_prompt = """You are a strict internal Compliance Auditor.
    Evaluate the proposed customer support resolution.

    RULES:
    1. The 'proposed_refund' CANNOT exceed the item's original cost.
    2. The email tone must not admit legal liability, only apologize for the inconvenience.
    3. The email must explicitly state the refund amount.

    If it violates ANY rule, return is_compliant=False and provide exact instructions on how to fix it.
    """

    user_prompt = """
    CRM Data: {crm}
    Proposed Refund: ${refund}
    Draft Email: {email}
    """

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", user_prompt)
    ])

    structured_llm = llm.with_structured_output(ComplianceOutput)
    chain = prompt | structured_llm

    result: ComplianceOutput = chain.invoke({
        "crm": state["crm_data"],
        "refund": state["proposed_refund"],
        "email": state["draft_email"]
    })

    if not result.is_compliant:
        print(f"   ❌ FAILED: {result.feedback_for_writer}")
    else:
        print("   ✅ PASSED COMPLIANCE")

    return {
        "compliance_passed": result.is_compliant,
        "compliance_feedback": result.feedback_for_writer
    }

def human_review_node(state: TicketState):
    # The graph will literally pause BEFORE hitting this node.
    # When it resumes, this prints out.
    print(f"👨‍💼 HUMAN IN THE LOOP: Manager approved the ${state['proposed_refund']} refund.")
    return state

def execute_node(state: TicketState):
    print(f"🚀 SYSTEM: Firing Stripe API for ${state['proposed_refund']}... Sending Email via SendGrid...")
    return state

# ==========================================
# 4. ROUTING LOGIC
# ==========================================
def compliance_router(state: TicketState) -> str:
    # 1. Infinite loop protection (escalate to human if AI can't get it right)
    if state["revision_count"] >= 3:
        print("⚠️ MAX REVISIONS REACHED. Routing to human.")
        return "human_review"

    # 2. Loop back if compliance failed
    if not state["compliance_passed"]:
        return "draft"

    # 3. High-Value routing (Manager Approval required for > $100)
    if state["proposed_refund"] > 100.00:
        print(f"⚠️ HIGH VALUE REFUND (${state['proposed_refund']}). Routing to Manager.")
        return "human_review"

    # 4. Auto-resolve
    return "execute"

# ==========================================
# 5. BUILD & COMPILE GRAPH
# ==========================================
workflow = StateGraph(TicketState)

workflow.add_node("fetch", fetch_data_node)
workflow.add_node("draft", adjudicate_and_draft_node)
workflow.add_node("compliance", compliance_node)
workflow.add_node("human_review", human_review_node)
workflow.add_node("execute", execute_node)

workflow.add_edge(START, "fetch")
workflow.add_edge("fetch", "draft")
workflow.add_edge("draft", "compliance")

# The complex decision point
workflow.add_conditional_edges(
    "compliance",
    compliance_router,
    {
        "draft": "draft",               # CYCLE
        "human_review": "human_review", # HITL
        "execute": "execute"            # AUTOMATION
    }
)

workflow.add_edge("human_review", "execute")
workflow.add_edge("execute", END)

# Compile with memory and a breakpoint BEFORE human review
memory = MemorySaver()
app = workflow.compile(checkpointer=memory, interrupt_before=["human_review"])

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# Thread config is REQUIRED when using memory. This represents this specific ticket.
config = {"configurable": {"thread_id": "ticket-9912"}}

initial_state = {
    "ticket_id": "ticket-9912",
    "customer_issue": "I just received my Premium Espresso Machine and it's completely shattered! I'm furious, I want a refund right now!"
}

print("=== STARTING TICKET PROCESSING ===")
# app.stream yields the output of each node as it finishes
for event in app.stream(initial_state, config):
    pass

# Because the refund will be $250 (which is > $100), the graph will pause.
print("\n=== 🛑 GRAPH PAUSED FOR HUMAN REVIEW ===")
state_snapshot = app.get_state(config)
print(f"Draft Email Pending Approval:\n{state_snapshot.values['draft_email']}")

# ... Time passes. A manager reviews the UI and clicks "Approve".
# We tell LangGraph to resume the exact same thread by passing None as the state.
print("\n=== ▶️ RESUMING WORKFLOW (MANAGER APPROVED) ===")
for event in app.stream(None, config):
    pass